# Design principles

> The rules comm-core is built by, why, and what they mean when you use or extend it.

In [ ]:
#| hide
import math

import numpy as np

from comm_core import *

comm-core models communication between agents that learn. Every part of it follows a small set of rules. This page
explains each rule, why it is there, and what it means for you, with a small example where it helps.
[Architecture](design.ipynb) describes the pieces themselves, and the examples show them at work.

1. [Communication is modelled, learning is not](#communication-is-modelled-learning-is-not)
2. [One abstraction, one question](#one-abstraction-one-question)
3. [Records flow, components act](#records-flow-components-act)
4. [Compose small pieces](#compose-small-pieces)
5. [Fidelity is a choice you can change late](#fidelity-is-a-choice-you-can-change-late)
6. [What happens at the same time is simulated together](#what-happens-at-the-same-time-is-simulated-together)
7. [Time is explicit](#time-is-explicit)
8. [The application owns the world](#the-application-owns-the-world)
9. [Reproducible by construction](#reproducible-by-construction)
10. [Usable inside and outside a network](#usable-inside-and-outside-a-network)
11. [A light core, heavy backends optional](#a-light-core-heavy-backends-optional)
12. [Validated, with the limits written down](#validated-with-the-limits-written-down)

## Communication is modelled, learning is not

comm-core knows nothing about learning. There are no observations, policies, gradients, agents or rewards in it.
A payload is opaque: it is carried, delayed, corrupted or lost, but never interpreted. A `Node` is passive, and
it neither sends nor receives on its own.

**Why:** the same questions come up in MARL, multi-agent world models and federated learning. Can these parties
talk? What gets through, when, and at what cost? Keeping learning out means one communication model serves all
of them, and a result about communication in one setting carries over to the others.

**What it means for you:** your learning code makes three decisions, and comm-core makes the rest.

| your code decides | comm-core decides |
|---|---|
| **when** to communicate | **whether** the message can go (is there a link?) |
| **what** to send | **when** it goes out (medium access) |
| **to whom** | **what** arrives (losses, bit errors, interference, delays) |
| | **what it cost** (bits, latency, energy) |

An adapter between your framework and comm-core stays thin. stable-marl's `stable_marl.comm`, for example, only
turns observations into messages and received messages into observations.

## One abstraction, one question

Each abstraction answers exactly one question, and nothing else:

| abstraction | its question |
|---|---|
| `Topology` | Who can communicate with whom? |
| `Protocol` | When is each message transmitted? |
| `CommunicationBackend` | How are the transmissions of a moment simulated, together? |
| `Channel` | What happens to one transmission? |
| `Fading` | What is the channel coefficient between two nodes, now? |
| `PowerControl` | How much power does a transmission use, and is it sent at all? |
| `ErrorModel` | How likely is a packet to be lost at a given SINR? |
| `Codec`, `Modulation` | How does a payload become bits, and bits symbols? |
| `MetricsCollector` | What did it cost? |
| `Network` | (ties them together) |

**Why:** a question with one owner has one place to change. A study usually varies one thing (the channel, the
protocol, the range) and keeps the rest fixed, so it changes exactly one object.

**What it means for you:** to compare options, build the network several times with one piece changed. Here the
same message traffic goes over three channels:

In [ ]:
def delivered(channel, steps=200):
    "Fraction of messages delivered between two agents, over `channel`."
    net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(channel))
    net.reset(seed=0)
    for _ in range(steps):
        net.send(Message('a', 'b', payload=None, size_bits=256))
        net.step()
    return net.metrics.summary().successful / steps

for name, channel in [('perfect', IdentityChannel()),
                      ('30% packet loss', AWGNChannel(snr_db=5, packet_error_rate=0.3)),
                      ('radio at 6 dB SNR', SystemLevelChannel(noise_power=10 ** -0.6))]:
    print(f'{name:18s} {delivered(channel):.2f} delivered')

perfect            1.00 delivered
30% packet loss    0.74 delivered
radio at 6 dB SNR  0.53 delivered


When you add something, decide which question it answers. If it answers two, it is probably two pieces.
`FragmentingChannel`, for example, only answers "how does a long message get through, packet by packet?". The
inner channel still decides what happens to each packet.

## Records flow, components act

Three plain records travel through the network:

* `Message`: what a node wants to say.
* `Transmission`: one attempt to deliver it.
* `TransmissionResult`: what happened to that attempt.

Components (topology, protocol, backend, channel) hold the behaviour and the state. Records only carry data.

* **Several attempts per message.** A message can be transmitted several times (retries), so attempts are records
  of their own.
* **Results carry measurements.** A result reports SNR, error rate, latency and energy, and its `metadata` holds
  what the model computed: the channel gain, transmit power, MCS, bit errors, number of packets, whether it was an
  outage, or interference. Analysis can happen afterwards, from the results, without instrumenting the model.
* **The receiver gets what was delivered.** The inbox holds the *result's* payload, not the message's. A bit-level
  channel delivers the payload with its bit errors.

**What it means for you:** read `metadata` to see why something happened. A new model should report its
quantities there too.

In [ ]:
channel = SystemLevelChannel(power_control=ChannelInversion(target_snr_db=10, max_power=20.0), noise_power=1.0)
for gain in (1.0, 0.1):                                       # a good link, and one needing too much power
    channel.fading = NoFading(math.sqrt(gain))
    t = Transmission(Message('a', 'b', 'hello', size_bits=64), 'a', 'b', start_time=0.0)
    r = channel.transmit(t, Link('a', 'b'))
    print(f'|h|² = {gain}: delivered {r.success}, metadata {r.metadata}')

|h|² = 1.0: delivered True, metadata {'channel_gain_db': 0.0, 'tx_power': 10.0}
|h|² = 0.1: delivered False, metadata {'channel_gain_db': -10.0, 'tx_power': 0.0, 'outage': True}


## Compose small pieces

Models are built by plugging small pieces into each other, not by setting flags on a large one:

* A fading goes into a channel; `GilbertElliottFading` wraps another fading.
* An error model and a power control go into a `SystemLevelChannel`.
* A channel goes into a `FragmentingChannel`, or into a backend.

**Why:** a large configurable model fixes the combinations its author thought of. Small pieces combine in ways
nobody planned. The robot-control example stacks ray-traced paths, bursts, a Wi-Fi error model and packet
retries, and none of those pieces knows about the others.

**What it means for you:** a custom model is usually one new small piece next to existing ones. Here is a Wi-Fi
link with bursts, built in a few lines:

In [ ]:
wifi = FragmentingChannel(                                              # packets of 1500 bytes, up to 7 retries
    SystemLevelChannel(
        fading=GilbertElliottFading(NoFading(1e-3), penalty_db=30, seed=0),   # bursts of interference
        noise_power=1e-11,
        per_model=EESMErrorModel(beta=28.0, threshold_db=21.0),         # an 802.11ac MCS (approximate)
    ),
    mtu_bits=12_000, retry_limit=7, overhead_s=200e-6, rate_bps=65e6,
)
image = Transmission(Message('robot', 'ap', None, size_bits=8 * 150_528), 'robot', 'ap', start_time=0.0)
r = wifi.transmit(image, Link('robot', 'ap'))
print(f"delivered {r.success} after {r.metadata['attempts']} attempts of {r.metadata['packets']} packets, "
      f"{1e3 * r.latency:.1f} ms")

delivered True after 101 attempts of 101 packets, 38.7 ms


## Fidelity is a choice you can change late

The same scenario can be simulated at several levels of detail:

| level | what is simulated | relative cost |
|---|---|---|
| `IdentityChannel`, `AWGNChannel` | nothing, or a fixed loss | negligible |
| `SystemLevelChannel` | SINR from the link budget, then a packet error rate (BPSK, threshold, EESM, or 5G NR from Sionna SYS) | small |
| `LinkLevelChannel` | every bit, uncoded BPSK | larger |
| `SionnaLinkChannel` | every bit of 5G NR: LDPC with a CRC, QAM, OFDM subcarriers | large (tens of ms per message on a CPU) |

The radio channels share one base, `RadioChannel`. It computes the *link budget* the same way at every level: the
channel from the fading, the power from the power control, and the SINR. So moving between levels changes only
the channel class, and the levels can be checked against each other.

**Why:** the right level depends on the question and the budget. Training needs speed. Checking a result needs
detail. A study should be able to start fast and confirm in detail without being rewritten.

**What it means for you:** develop and train at the system level, then confirm important results at the link
level with the same fading, power control and protocol. The [levels of fidelity](examples/04_fidelity.ipynb)
example shows how far apart they are.

## What happens at the same time is simulated together

Each round of a `Network` hands *all* of that moment's transmissions to the backend at once
(`CommunicationBackend.transmit_all`). The roles are split:

* **The protocol** decides which transmissions share a moment.
* **The backend** decides how they affect each other.
* **The channel** decides what happens to each.

**Why:** two things need the whole moment.

* **Interference.** A transmission's SINR depends on everything else on the air at the same time
  (`InterferenceBackend`).
* **Speed.** Heavy models are faster in batches: the system and link levels process a round's transmissions in
  one call (`transmit_batch`), e.g. one Sionna call per round.

**What it means for you:** collisions and their avoidance are modelled, not assumed away. Sending everything at
once with `IdealProtocol` over an `InterferenceBackend` means nobody hears; a medium access protocol (TDMA, ALOHA)
fixes that. See the [medium access](examples/02_medium_access.ipynb) example.

## Time is explicit

The network has a clock. Each move of the clock (`step`, `step_to`) ends with a round of transmissions. On top of
that:

* `send(message, at=t)` sends a message at any time `t`, with a round at exactly `t`.
* With `delayed_delivery=True`, a message arrives at its transmission's start + latency. The latency covers the
  airtime of its packets, their retries and the overheads.
* A transmission carries its time (`start_time`), so time-varying models can use it: bursts
  (`GilbertElliottFading`), or the state at each retry of a packet.

**Why:** in many studies, *when* information arrives matters as much as *whether*: stale messages, a robot
waiting for actions, a model update arriving after the round closed. If time stayed implicit, every model would
reinvent it.

**What it means for you:** simple uses need none of this. One `step()` per environment step, with messages
arriving within the step, is the default. Studies of delay switch on delayed delivery and send at the times
things happen.

In [ ]:
class Airtime(Channel):
    "Everything arrives, after 30 ms."
    def transmit(self, transmission, link):
        return TransmissionResult(success=True, transmission=transmission, payload=transmission.message.payload, latency=0.03)

net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(Airtime()),
              delayed_delivery=True)
net.send(Message('a', 'b', 'request'), at=0.012)             # sent at 12 ms
for t in (0.02, 0.04, 0.05):
    net.step_to(t)
    print(f't = {1e3 * t:.0f} ms: b has {[(m.payload, round(1e3 * m.metadata["received_at"])) for m in net.receive("b")]}')

t = 20 ms: b has []
t = 40 ms: b has []
t = 50 ms: b has [('request', 42)]


## The application owns the world

comm-core never moves nodes, never decides who is near whom, and keeps no map of the world. The application
tells it:

* **Positions:** `set_positions` on a channel. It passes them on to the fadings that need them, e.g.
  `RayTracedPathsFading`.
* **Changing neighbourhoods:** a new topology, e.g. `DistanceTopology` from the positions.

**Why:** the environment already simulates the world. A second copy inside comm-core would drift from it, and
would tie comm-core to one kind of world (grids, continuous spaces, robots).

**What it means for you:** before each round, give the channel the positions from your environment. stable-marl's
`CommunicationWrapper` does this at every step. Discrete and continuous worlds are handled the same way.

## Reproducible by construction

* **Own generators:** every random component has its own generator (channels, fadings, `SlottedALOHA`).
  comm-core never touches the global `random` or `numpy.random` state.
* **Seeding:** `Network.reset(seed)` reseeds the protocol and the backend. The backend reseeds its channel, which
  reseeds its fading, each with a seed derived from the one given.
* **Common random numbers:** bursts and fades are fixed by their seeds, not by the order of events. Two methods
  run with the same seed therefore meet the same bursts, and their difference comes from the method, not from
  luck.

**Why:** communication adds randomness to learning experiments that are already noisy. If it can't be repeated,
it can't be compared or debugged.

In [ ]:
def episode(seed):
    net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), SlottedALOHA(p=0.5),
                  AnalyticalBackend(SystemLevelChannel(fading=RayleighFading(), noise_power=0.1)))
    net.reset(seed)
    out = []
    for _ in range(30):
        net.send(Message('a', 'b', None, size_bits=128))
        out += [r.success for r in net.step()]
    return out

print('same seed, same episode:', episode(0) == episode(0), '| another seed, another episode:', episode(0) != episode(1))

same seed, same episode: True | another seed, another episode: True


## Usable inside and outside a network

The `Network` handles one message at a time, as Python objects, which is convenient for experiments and
environments. Training often needs whole batches of links at once, e.g. thousands of agent pairs × time steps
inside a model. The pieces underneath are plain vectorized functions on numpy arrays or torch tensors (on any
device): codecs, `BPSK`, `propagate`, `PowerControl.allocate`, `eesm`.

**What it means for you:** the same channel model can sit inside a training graph, batched, and inside an
environment, per message. See `link` for a batched example.

In [ ]:
codec, bpsk = IndexCodec(256), BPSK()
csi = (np.random.default_rng(0).standard_normal((1024, 3, 1)) + 1j * np.random.default_rng(1).standard_normal((1024, 3, 1))) / np.sqrt(2)
messages = np.random.default_rng(2).integers(0, 256, (1024, 49))           # 1024 samples, 49 codebook indices each
power, sent = ChannelInversion(target_snr_db=6, max_power=10).allocate(np.abs(csi) ** 2, 1.0)
received = codec.decode(bpsk.demodulate(propagate(bpsk.modulate(codec.encode(messages[:, None])), csi, power, 1.0), csi))
print(received.shape, f'{np.mean(sent):.0%} of the links used,',
      f'{np.mean((received != messages[:, None])[sent[..., 0]]):.1%} of their indices corrupted')

(1024, 3, 49) 67% of the links used, 1.9% of their indices corrupted


## A light core, heavy backends optional

The core needs only numpy. Heavier models are optional extras, imported only when used:

* **Sionna SYS and PHY** (PyTorch): `pip install 'comm-core[sionna]'`.
* **Sionna RT**, for tracing scenes.

Tests that need an extra are flagged (`#| sionna`, `#| slow`) and run only on request.

**Why:** a communication model is often added to a project that already has its own heavy dependencies, and it
should not drag in more than it uses. Ray tracing, in particular, runs where the GPUs are, once. Its results are
saved, and used anywhere afterwards without Sionna.

## Validated, with the limits written down

Every model is checked against an independent reference, and the checks are part of the tests or the examples:

| model | checked against | result |
|---|---|---|
| `BPSK`, `propagate` | the analytic BPSK bit error rate | within the tests' tolerance (up to 10%) at 0–7 dB |
| `SystemLevelChannel` | `LinkLevelChannel` with a CRC | packet loss within 0.04 at 4–8 dB |
| `RayTracedPathsFading` | exact geometry; direct Sionna RT traces | exact for line of sight and single reflections; −47 to −62 dB error with a fixed access point |
| `SionnaErrorModel` | `SionnaLinkChannel` (bit-level NR) | within 0.001 where the tables have data (50 decoder iterations); 0.5–1 dB apart between table points |
| `InterferenceBackend` | hand-computed SINRs | exact |

Where a model is known to be wrong, the page says so, with numbers. For example:

* Sionna SYS is far too optimistic for messages under about 100 bits.
* Moved paths fail where a reflection appears or disappears between anchors.
* Interference is treated as Gaussian noise.
* There is no HARQ combining.

**What it means for you:** before relying on a model for a claim, read its page's limits, and check your regime
(message sizes, SINR range, anchor spacing) against the validations. A new model should come with its own check.

## Extending comm-core

A checklist for a new piece:

1. **Which question does it answer?** Subclass that abstraction (see the table under *One abstraction, one
   question*, and *Extending* in [Architecture](design.ipynb)).
2. **Does it need the moment, or one transmission?** Interactions between transmissions go in a backend; the
   fate of one transmission goes in a channel.
3. **Does it depend on time or position?** Read `transmission.start_time`; take positions through
   `set_positions`.
4. **Is it random?** Give it its own generator and a `reset(seed)`.
5. **What does it measure?** Report it in the result's `metadata`.
6. **What is it checked against?** Add the check, and write down where it stops being right.